# **Step 6: Deep Dive into LoRA**

#**Research/Learn**:
LoRA theory in depth (low-rank adaptation, matrix
decomposition, parameter efficiency, target modules, mathematical foundation).

#**Low-rank adaptation (LoRA)**  
LoRA is a technique used to adapt machine learning models to new contexts. It can adapt large models to specific uses by adding lightweight pieces to the original model rather than changing the entire model. A data scientist can quickly expand the ways that a model can be used rather than requiring them to build an entirely new model.
#Matrix Decomposition
LoRA apply matrix decomposition technique. By that way, LoRA is built on the understanding that large models inherently possess a low-dimensional structure. By leveraging smaller matrices, which are called low-rank matrices, LoRA adapts these models effectively. This method focuses on the core concept that significant model changes can be represented with fewer parameters, thus making the adaptation process more efficient.

**#Mathematical Foundation**
A layer have formula: y=Wx(W:weighed matrix, x :input batch, y:label). And tradditional fine tune model add delta_W to original Weighed, it's spend alot of time and GPU. And with LoRA technique have formula: y=Wx+ BAx. it  freeze originial.

**#Parameter Efficiency**

One of the biggest advantages of LoRA is parameter efficiency. Traditional fine-tuning updates all parameters of the model, which requires huge GPU memory and training cost. In contrast, LoRA only trains the low-rank matrices A and B while freezing the original pretrained weights.

Instead of training a full matrix: A*B,LoRA only trains: A*r + r*B

where:
* r is a very small rank
* r≪A,B

This significantly reduces the number of trainable parameters and memory usage.
For example, a 4096*4096 matrix contains more than 16 million parameters, while LoRA with rank r=8 only trains around 65 thousand parameters.

**#Target Modules**

LoRA is usually applied to specific modules inside Transformer architectures instead of the entire model. These modules are called target modules.

Common target modules include:

Query projection (Wq​)
Key projection (Wk)
Value projection (Wv)
Output projection (Wo​)

These layers are important because they directly affect the attention mechanism of the Transformer.


#**Action**

In [1]:
!pip install peft trl torchao>=0.16.0 --upgrade

In [2]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# 2. DATASET

dataset = load_dataset("fawern/Text-to-sql-query-generation")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/277 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [4]:
print(dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>'}


In [5]:

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
import re

def format_chat(example):

    text = example["prompt"]

    # lấy phần [INST] ... [/INST]
    inst_text = re.search(
        r"\[INST\](.*?)\[/INST\]",
        text,
        re.S
    ).group(1).strip()

    # tách system + question
    parts = inst_text.split("Question :")

    system_message = parts[0].strip()
    question = parts[1].strip()

    # lấy SQL query
    query = re.search(
        r"SQL Query :(.*?)</s>",
        text,
        re.S
    ).group(1).strip()

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": query
        }
    ]

    return {"messages": messages}


In [7]:
chat= format_chat(dataset['train'][0])
print(chat)

{'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}]}


In [8]:
formated_dataset = dataset.map(format_chat)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
def formatting(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}
Qwen_template_dataset=formated_dataset.map(formatting)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [10]:

print(Qwen_template_dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>', 'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}], 'text': '<|im_start|>system\nYou are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>\n<|im_start|>user\nName the home team for carlton away team<|im_end|>\n<|im_start|>assistant\nSELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>\n'}


In [11]:
Qwen_template_dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'messages', 'text'],
        num_rows: 10000
    })
})

In [12]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_ds = Qwen_template_dataset.map(tokenize)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [13]:
tokenized_ds = tokenized_ds.map(

    remove_columns=[col for col in tokenized_ds["train"].column_names if col not in ["input_ids", "attention_mask"]]
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
token= tokenized_ds['train'][0]
token

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  675,
  279,
  2114,
  2083,
  369,
  1803,
  75,
  777,
  3123,
  2083,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  2114,
  26532,
  4295,
  1965,
  1269,
  62,
  22,
  22,
  5288,
  3123,
  26532,
  284,
  330,
  6918,
  75,
  777,
  1,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [15]:
print(tokenizer.decode(token['input_ids']))

<|im_start|>system
You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>
<|im_start|>user
Name the home team for carlton away team<|im_end|>
<|im_start|>assistant
SELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>



In [16]:
dataset = tokenized_ds["train"].train_test_split(
    test_size=0.1,
    seed=42
)

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [18]:
dataset['train'][0]

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  3838,
  374,
  279,
  220,
  17,
  15,
  16,
  16,
  13369,
  5264,
  323,
  264,
  220,
  17,
  15,
  16,
  15,
  1207,
  37,
  30,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  330,
  17,
  15,
  16,
  16,
  1,
  4295,
  1965,
  62,
  16,
  17,
  22,
  23,
  22,
  5288,
  330,
  51,
  9783,
  1,
  284,
  364,
  2863,
  495,
  10480,
  1787,
  6,
  3567,
  330,
  17,
  15,
  16,
  15,
  1,
  284,
  364,
  80,
  69,
  6,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

#Action 1: LoRA RANK 16

In [22]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=400,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


Step,Training Loss,Validation Loss
100,0.943197,0.948525
200,0.908128,0.865704
300,0.830362,0.825707
400,0.812164,0.803718
500,0.764841,0.787737
600,0.726247,0.776583
700,0.734569,0.770163
800,0.683008,0.766582


TrainOutput(global_step=846, training_loss=0.8383678133042428, metrics={'train_runtime': 1915.284, 'train_samples_per_second': 14.097, 'train_steps_per_second': 0.442, 'total_flos': 8633232530841600.0, 'train_loss': 0.8383678133042428, 'epoch': 3.0})

In [ ]:
from huggingface_hub import login

login()
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_16"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_16"
)

#Prediction

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ===== Base model =====
base_model = "Qwen/Qwen2.5-0.5B-Instruct"

# ===== LoRA adapter path =====
lora_path = "./qwen_sql_lora/checkpoint-2815"

# ===== Load tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(base_model)

# ===== Load base model =====
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ===== Load LoRA adapter =====
model = PeftModel.from_pretrained(
    model,
    lora_path
)
# ===== Inference mode =====
model.eval()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Show all customers from Vietnam
assistant
SELECT DISTINCT customer_id FROM customers WHERE location = 'Vietnam'


In [ ]:


# ===== Test question =====
messages = [
    {
        "role": "system",
        "content": " You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question."
    },
    {
        "role": "user",
        "content": "Show all customers from Vietnam"
    }
]


# ===== Apply Qwen chat template =====
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# ===== Tokenize =====
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ===== Generate =====
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=False
    )

# ===== Decode =====
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Show all customers from Vietnam
assistant
SELECT DISTINCT customer_id FROM customers WHERE location = 'Vietnam'


In [ ]:
message_2 = [
    {
        "role": "system",
        "content": " You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question."
    },
    {
        "role": "user",
        "content": "Users with highest reputation both in SO and Math ( geometric mean = average digits)"
    }
]

In [ ]:

# ===== Apply Qwen chat template =====
text = tokenizer.apply_chat_template(
    message_2,
    tokenize=False,
    add_generation_prompt=True
)

# ===== Tokenize =====
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ===== Generate =====
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=False
    )

# ===== Decode =====
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Users with highest reputation both in SO and Math ( geometric mean = average digits)
assistant
SELECT DISTINCT LOWER(LOWER(username)) AS username, LOWER(LOWER(email)) AS email FROM users WHERE (1 + RAND() * 200) > (SELECT MAX(CASE WHEN t1.reputation >= 5 THEN Reputation ELSE 0 END) FROM (SELECT COUNT(*) AS "reputation", ROW_NUMBER() OVER (ORDER BY Reputation DESC) AS "rank" FROM users AS t1 JOIN user_reputation AS t2 ON t1.id = t2.user_id GROUP BY t1.id) AS q1) AS p1 JOIN (SELECT COUNT(*) AS "reputation", ROW_NUMBER() OVER (ORDER BY Reputation


#Action 2: Fine-tuned with LoRA RANK 4

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql_lora_4",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=400,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)

trainer.train()

Step,Training Loss,Validation Loss
100,1.063029,0.986433
200,0.902927,0.907461
300,0.863437,0.866992
400,0.863619,0.843362
500,0.855095,0.824124
600,0.799575,0.808742
700,0.785546,0.800245
800,0.800735,0.794524
900,0.746331,0.786732
1000,0.748668,0.781446


TrainOutput(global_step=1689, training_loss=0.814986523684134, metrics={'train_runtime': 2731.0491, 'train_samples_per_second': 9.886, 'train_steps_per_second': 0.618, 'total_flos': 7037614326403584.0, 'train_loss': 0.814986523684134, 'epoch': 3.0})

spend 5.9 RAM

In [ ]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp0c1mneoe/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  24%|##4       |  533kB / 2.19MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4/commit/f8aac3c78913cb7a3a7ff1547aaed860366ac12d', commit_message='Upload model', commit_description='', oid='f8aac3c78913cb7a3a7ff1547aaed860366ac12d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql-lora_rank_4'), pr_revision=None, pr_num=None)

Fine-tuned with LoRA Rank 16; took 12 RAM GPU and 45m for training 3 epoches. Loss: 0.76

#Action 3: Fine-tuned with LoRA RANK 32

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql_lora_32",

    per_device_train_batch_size=4,#
    gradient_accumulation_steps=8, #because LoRA take fewer RAM than fully fintune- so i raise 2 super parameter

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=400,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 4,325,376 || all params: 498,358,144 || trainable%: 0.8679


In [ ]:

trainer.train()

Step,Training Loss,Validation Loss
100,0.949782,0.954943
200,0.913050,0.868769
300,0.831280,0.827034
400,0.813590,0.803539
500,0.765139,0.789440
600,0.725613,0.776748
700,0.734528,0.770130
800,0.680752,0.766340


TrainOutput(global_step=846, training_loss=0.8397159587688762, metrics={'train_runtime': 1502.3998, 'train_samples_per_second': 17.971, 'train_steps_per_second': 0.563, 'total_flos': 8685087629746176.0, 'train_loss': 0.8397159587688762, 'epoch': 3.0})

In [ ]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_32"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_32"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9m8crgia/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         |  552kB / 17.3MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_32/commit/f97c2c5fe00041d1fadc525e370803b81f216146', commit_message='Upload model', commit_description='', oid='f97c2c5fe00041d1fadc525e370803b81f216146', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_32', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql-lora_rank_32'), pr_revision=None, pr_num=None)

Spend 14.5 RAM GPU

#Action 4: Fine-tuned with LoRA RANK 7 and target modules

In [ ]:
# ===== Load tokenizer =====


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=32,
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    # FFN:
    "up_proj",
    "down_proj",
    "gate_proj"
],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql_lora_4_target_7",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=400,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)

trainer.train()

trainable params: 2,199,552 || all params: 496,232,320 || trainable%: 0.4433


Step,Training Loss,Validation Loss
100,0.796984,0.818278
200,0.795068,0.757746
300,0.689361,0.730388
400,0.684857,0.716011
500,0.650019,0.707255
600,0.575775,0.701517
700,0.585263,0.700720
800,0.539530,0.695733


TrainOutput(global_step=846, training_loss=0.7003319581913328, metrics={'train_runtime': 1942.3497, 'train_samples_per_second': 13.901, 'train_steps_per_second': 0.436, 'total_flos': 8634116424572928.0, 'train_loss': 0.7003319581913328, 'epoch': 3.0})

In [ ]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4_target_module_7"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4_target_module_7"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpw5d6aztj/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 37.3kB / 8.84MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4_target_module_7/commit/9f31b9a91fbe3c7c63b759ee7db0c2018abce5c2', commit_message='Upload model', commit_description='', oid='9f31b9a91fbe3c7c63b759ee7db0c2018abce5c2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4_target_module_7', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql-lora_rank_4_target_module_7'), pr_revision=None, pr_num=None)

Spend 12.7 RAM GPU

#Action 5: Fine-tuned with LoRA RANK 4 and target modules(4 and FFN)

In [ ]:
# ===== Load tokenizer =====


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=32,

    target_modules=[


        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql_lora_4_",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=400,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)

trainer.train()

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


Step,Training Loss,Validation Loss
100,0.976227,0.979899
200,0.947734,0.900006
300,0.874428,0.862587
400,0.857276,0.841585
500,0.806985,0.828086
600,0.781966,0.817124
700,0.787929,0.809950
800,0.740145,0.806649


TrainOutput(global_step=846, training_loss=0.8884168646296147, metrics={'train_runtime': 1285.6835, 'train_samples_per_second': 21.001, 'train_steps_per_second': 0.658, 'total_flos': 8587859319300096.0, 'train_loss': 0.8884168646296147, 'epoch': 3.0})

Spend: 12 RAM GPU

In [ ]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4_target_module_2"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_4_target_module_2"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpz2g1hbfe/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  48%|####8     |  529kB / 1.09MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4_target_module_2/commit/13c061959b4920504dba7deed80526b7fc0aa95b', commit_message='Upload model', commit_description='', oid='13c061959b4920504dba7deed80526b7fc0aa95b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_4_target_module_2', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql-lora_rank_4_target_module_2'), pr_revision=None, pr_num=None)


#REVIEWS:
With complex or hard SQL queries, the model tends to hallucinate and generate incorrect tables, columns, or query structures. This happens because the model only has 0.5B parameters, which limits its reasoning and schema understanding capabilities. While the model performs well on simple SQL generation tasks, it struggles with multi-table joins, nested queries, aggregation logic, and cross-domain reasoning. Smaller language models mainly learn token patterns rather than deep semantic understanding, so they are more likely to hallucinate when the query becomes complicated.

After evaluating 5 different LoRA fine-tuning with the same batch_size configurations, the results are summarized below:

* **Rank=4, 4 target modules**: Consumed 5.9 GB of GPU VRAM. Final train loss: 0.735; final test loss: 0.761. Total trainable adapter parameters: 540,672 (~0.109%).

* **Rank=16, 4 target modules**: Consumed 12.5 GB of GPU VRAM. Final train loss: 0.680; final test loss: 0.770. Total trainable adapter parameters: 2,162,688 (~0.436%).

* **Rank=32, 4 target modules**: Consumed 14.5 GB of GPU VRAM. Final train loss: 0.735; final test loss: 0.761. Total trainable adapter parameters: 4,325,376 (~0.868%).

* **Rank=4, 7 target modules (Attention and FFN)**: Consumed 12.7 GB of GPU VRAM. Final train loss: 0.540; final test loss: 0.696. Total trainable adapter parameters: 2,199,552 (~0.443%).

* **Rank=4, 2 target modules(v,o)**: Consumed 12.0 GB of GPU VRAM. Final train loss: 0.740; final test loss: 0.810. Total trainable adapter parameters: 270,336 (~0.055%).

=> With text to SQL dataset on small LLM. Optimization stategy don't increase matrix rank, should expand module target for increase logic inference ablity of model.